<img src="https://cads.ai/image/CADSLogo.png" style="height: 100px;" align=left>
<div align=right><svg height=100px viewBox="0 0 24 24" fill="none" xmlns="http://www.w3.org/2000/svg"><g clip-path="url(#clip1_4783_495)"><path d="M6.64803 14.103C7.89438 12.8566 8.595 11.1643 8.595 9.40177C8.595 7.63928 7.89378 5.94697 6.64803 4.70058L1.94697 0C0.701225 1.24639 0 2.9387 0 4.70119C0 6.46368 0.701225 8.15599 1.94697 9.40238L6.64742 14.103H6.64803Z" fill="#7B9AB5"></path><path d="M16.4845 14.5379C15.2388 13.2921 13.5459 12.5908 11.7841 12.5908C10.0222 12.5908 8.32936 13.2921 7.08301 14.5379L11.7841 19.239C13.0298 20.4848 14.7227 21.1861 16.4851 21.1861C18.2476 21.1861 19.9398 20.4848 21.1862 19.239L16.4851 14.5379H16.4845Z" fill="#7B9AB5"></path><path d="M1.95832 19.228C3.20468 20.4738 4.89693 21.1751 6.65938 21.1751V14.5269H0.0107422C0.0113472 16.2893 0.711968 17.9817 1.95832 19.228Z" fill="#7B9AB5"></path><path d="M18.2997 7.58717C17.0533 6.34138 15.3611 5.63953 13.598 5.64014C11.8356 5.64014 10.1433 6.34138 8.89697 7.58777L13.598 12.289L18.2997 7.58717Z" fill="#7B9AB5"></path></g></svg></div>

<a box-sizing="border-box" color-scheme=dark font-family="IBM Plex Sans" line-height=30px font-size=72px font-weight=700 align=center>© 2026 — Mehdi Lotfinejad ⨯ Azhar bin Ab Halim @ DATATWEETS</a>

# Prompts that behave: templates, roles and examples
Day 1 | Notebook 2 of 4

![On the left a Seri Maju claims letter has yellow dashed blanks for name, item, amount and status, like a mail-merge stencil. On the right three example cards show the style, with an arrow pointing to the letter.](https://github.com/saifulak/EAE/blob/main/LLM/media/d1-02-hero-prompt-stencil.svg?raw=1)

**You can re-run any practice until it works. Nothing here is recorded or marked.**

**Today in one sentence:** you teach the helper to answer in a fixed shape, using prompt templates, roles and a few examples.

## Today's story

The helpdesk at Seri Maju Berhad (fictional; Kuala Lumpur, Penang and Johor Bahru sites) gets many expense claims. Staff want a quick answer — "who must approve my claim?" Free-form model replies are chatty and hard to use. In this notebook you teach the model to answer in one fixed shape, using a **template**, **roles** and **examples**.

*Amounts are in ringgit (RM) and invented. *Fictional example, not legal or tax advice.*

## What you will be able to do

3. Build a prompt template and a chat template, add worked examples (few-shot), and compare with a plain instruction.

---

**Prerequisites:** `d1-01-*.ipynb` helps (calling `get_llm()`). Basic Python `str`ings and `dict`s.

**Words you do not know?** See the [Glossary](glossary.md).

In [1]:
# Answer switch (change it, then choose Run > Restart Kernel and Run All Cells).
#     False = you type the blanks.
#     True = Run All reads the reference answers.
USE_SOLUTIONS = globals().get("USE_SOLUTIONS", False)

In [2]:
%pip install "langchain-groq>=0.3,<0.4"

In [3]:
# ---- CANONICAL SETUP ----
# ---- HELPER - Run it, then ignore it. ----



## You can re-run any practice until it works. Nothing here is recorded or marked.
# Standard-library helpers for settings files, timing and folder paths.
import json
import os
import sys
import time
from glob import glob
from pathlib import Path
from typing import Literal

# Third-party pieces: HTTP client, Groq chat model, and LangChain base classes for models and embeddings.
import httpx
from dotenv import load_dotenv
from langchain_core.caches import InMemoryCache
from langchain_core.embeddings import Embeddings
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_groq import ChatGroq

# scikit-learn powers the offline TF-IDF embeddings; stop early with an install hint if it is missing.
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
except ImportError as e:
    raise ImportError(
        "scikit-learn is needed for TfidfEmbeddings. In a terminal run: "
        "pip install scikit-learn  (then restart the kernel)."
    ) from e

# Load a default .env file, if there is one, into the environment.
load_dotenv()


SOLUTIONS = {}  # step_id -> (builder function, one-sentence why, code text shown by show_solution)
CHECKPOINTS = {}  # part_id -> function(ns) that rebuilds the variables a late learner needs (canned/mock only)
CODES = {}  # part -> reference code texts, in order

# Look up per-million-token prices for the chosen GROQ_MODEL, used for cost estimates.
# MYR is preferred, then USD, then a built-in default.
with open("_groq_model_per_million_prices.json", mode="rb") as _model_prices_json:
    _MODEL_PRICES = json.load(_model_prices_json).get(
        os.environ.get("GROQ_MODEL", ""), {}
    )
    model_input_price = (_model_input_price := _MODEL_PRICES.get("input", {})).get(
        "MYR", _model_input_price.get("USD", 0.50)
    )
    model_output_price = (_model_output_price := _MODEL_PRICES.get("output", {})).get(
        "MYR", _model_output_price.get("USD", 2.00)
    )


def register(step_id, why, code=""):
    """Decorator used by the notebook: SOLUTIONS['d1-01-p2'] = the reference builder."""

    def deco(fn):
        SOLUTIONS[step_id] = (fn, why, code or getattr(fn, "__doc__", "") or "")
        return fn

    return deco


def checkpoint(part_id):
    """Decorator: register the function that rebuilds this Part's variables (see fast_forward)."""

    def deco(fn):
        CHECKPOINTS[part_id] = fn
        return fn

    return deco


def pick(step_id, attempt=None, is_using_solutions: bool = True):
    """Use your attempt, or the reference answer if USE_SOLUTIONS is True or your attempt is None."""
    if is_using_solutions or attempt is None:
        if step_id not in SOLUTIONS:
            raise KeyError(
                f"No reference answer registered for '{step_id}'. Click the heading 'Part A', "
                "choose Run > Run All Above Selected Cell, then run this cell again."
            )
        return SOLUTIONS[step_id][0]()
    return attempt


def check(step_id, ok, hint=""):
    """Plain self-check: tick or cross plus a hint. Unlimited re-runs. Nothing is stored or counted."""
    print(
        ("\u2714 " + step_id + ": looks right")
        if ok
        else (
            "\u2718 "
            + step_id
            + ": not yet. "
            + (hint or "Re-read the step, change your code, run again.")
        )
    )
    return bool(ok)


def show_solution(step_id):
    """Print the reference code and why it works. Keyboard-friendly: no copying needed, use pick()."""
    if step_id not in SOLUTIONS:
        print("No reference answer registered for", step_id)
        return
    _, why, code = SOLUTIONS[step_id]
    print(code.rstrip() or "(see the reference cell)")
    print("Why:", why)


def fast_forward(part_id):
    """Late or stuck? Rebuild this Part's variables from reference code (canned/mock, no API calls)."""

    if part_id not in CHECKPOINTS:
        print(
            "Not ready yet. Click the heading 'Part A', choose Run > Run All Above Selected Cell, "
            "then run this cell again. Known parts:",
            ", ".join(sorted(CHECKPOINTS)) or "(none registered yet)",
        )
        return

    if globals().get("LLM_MODE") == "groq":
        print("Note: fast-forward uses canned replies, not the live model.")

    CHECKPOINTS[part_id](globals())
    print("\u2714 Caught up to", part_id, "- continue with the next cell.")


# Load the workplace policy texts as system messages so every answer can be grounded in them.
def get_system_corpus(context: Literal["workplace"]) -> list[tuple[str, str]]:
    _corpus_dir: Path
    system_prompt: list[tuple[str, str]] = []

    if context == "workplace":
        _corpus_dir = Path.cwd() / "data" / "workplace*" / "*.txt"

    else:
        raise ValueError(f"Unsupported corpus: '{context}'")

    # Read each policy .txt file under data/workplace*/ as one system message.
    for _corpus_pathname in glob(str(_corpus_dir)):
        with open(_corpus_pathname, "r", encoding="utf-8") as _corpus_file:
            system_prompt.append(("system", _corpus_file.read()))

    # Finish with a rule that makes answers cite their source identifier.
    system_prompt.append(
        (
            "system",
            "Assistant should always cite source identifier to support claim when answering any asks.",
        )
    )

    return system_prompt


# Remember a code snippet under its Part so a late learner can rebuild that Part.
def state(part, code):
    CODES.setdefault(part, []).append(code.strip())


# Register a reference answer: keep its code text and a builder that runs it and returns result.
def ref(part, step, why, code):
    code = code.strip()
    if part:
        CODES.setdefault(part, []).append(code)

    # Run the reference code in a copy of the notebook variables and hand back its result.
    def build():
        ns = dict(globals())
        exec(code, ns)
        return ns["result"]

    register(step, why, code)(build)


# Define how to catch up: re-run the stored code of the listed Parts in order (used by fast_forward).
def rebuild(part, *parts):
    checkpoint(part)(lambda ns: [exec(c, ns) for p in parts for c in CODES[p]])


# Build the shared system messages once so later cells can reuse them.
SYSTEM_PROMPT = get_system_corpus("workplace")


# Guard the Python version: 3.14 or newer is too new for some course packages.
if sys.version_info[:2] >= (3, 14):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} is too new for this course: some packages are not ready. "
        "Create the environment with Python 3.12 (3.11-3.13 work) and pick it as the notebook kernel."
    )

# Also stop on versions older than the supported 3.11.
assert (3, 11) <= sys.version_info[:2] <= (3, 13), (
    f"Use Python 3.11-3.13 (you have {sys.version.split()[0]})"
)
# Key lookup order:
#   1. file named in LM_ENV_FILE
#   2. ~/.langchain-mastery/.env (local, not synced)
#   3. .env next to notebooks.
# Only these four names are read from the file; everything else in it is ignored.

# Names read from the settings file, and folder names that suggest cloud-synced storage.
_ALLOWED = ("GROQ_API_KEY", "GROQ_MODEL", "LLM_MODE", "LM_ENV_FILE")
_SYNCED = ("onedrive", "sharepoint", "dropbox", "google drive", "googledrive", "icloud")


def _tilde(
    p,
):  # show the home folder as ~ so a shared screen does not show your user name
    p, home = str(p), str(Path.home())
    return (
        "~" + p[len(home) :]
        if p.lower().startswith(home.lower())
        else "(outside home) " + Path(p).name
    )


# Warn early if LM_ENV_FILE points to a file that does not exist.
_lm = os.getenv("LM_ENV_FILE", "").strip()
if _lm and not Path(_lm).is_file():
    print(
        "NOTE: LM_ENV_FILE is set but that file was not found, so it is ignored. Check the path."
    )

# Candidate settings files in lookup order; the first one that exists is used.
_env_files = [
    _lm,
    str(Path.home() / ".langchain-mastery" / ".env"),
    str(Path.cwd() / ".env"),
    str(Path.cwd() / "courseware" / ".env"),
]

# Copy only the allowed names from the chosen file into the environment, never overwriting values already set.
_env_file = next((p for p in _env_files if p and Path(p).is_file()), None)
if _env_file:
    try:
        from dotenv import dotenv_values

        for _k, _v in dotenv_values(_env_file).items():
            if (
                _k in _ALLOWED and _v is not None and _k not in os.environ
            ):  # existing variables win
                os.environ[_k] = _v

        # Tell the learner which file was used, with the home folder hidden.
        print(
            "Settings file:",
            _tilde(_env_file),
            "(only",
            ", ".join(_ALLOWED),
            "are read)",
        )

        # Warn if the file sits in a synced folder, where a key could be shared by accident.
        if any(w in _env_file.lower() for w in _SYNCED):
            print(
                "WARNING: your .env is in a synced folder (OneDrive, SharePoint, Dropbox, Google Drive "
                "or iCloud). Move it to ~/.langchain-mastery/.env"
            )

    except ImportError:
        print("python-dotenv not installed: using system environment variables only")
else:
    # With no settings file the notebook still runs in mock mode, so no key is needed.
    print("No .env file found: mock mode, no key needed.")

# Choose mock (offline, canned replies) or groq (live model); mock is the default.
LLM_MODE = os.getenv("LLM_MODE", "mock").strip().lower()
if LLM_MODE not in ("mock", "groq"):
    raise ValueError("LLM_MODE must be 'mock' or 'groq' (check your .env file)")

# Tracing switch: mock mode forces every tracing variable off; groq mode only warns if one is on.
_TRACING = ("LANGCHAIN_TRACING_V2", "LANGCHAIN_TRACING", "LANGSMITH_TRACING")
if LLM_MODE == "mock":
    for _t in _TRACING:
        os.environ[_t] = "false"

    print(
        "Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere)."
    )

elif any(os.getenv(_t, "").strip().lower() == "true" for _t in _TRACING):
    print(
        "WARNING: a LangChain tracing variable is on: live runs may be sent to a tracing service. Set it to false."
    )


class ScriptedToolChat(BaseChatModel):
    """Mock model for tool calling: replays a script. A str is a final answer; a dict
    {'name':..., 'args':{...}} (or a list of them) becomes AIMessage.tool_calls. Canned, not real."""

    # The script to replay and the position of the next step.
    script: list = []
    pos: int = 0

    @property
    def _llm_type(self):
        return "scripted-tool-chat"

    def bind_tools(self, tools, **kwargs):
        return self  # tools are ignored: the script decides

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        # Take the next scripted step, looping back to the start when the script runs out.
        step = (
            self.script[self.pos % len(self.script)] if self.script else "(mock reply)"
        )
        self.pos += 1

        # A text step becomes a final answer; a dict step becomes tool calls with made-up ids.
        if isinstance(step, str):
            msg = AIMessage(content=step)

        else:
            calls = step if isinstance(step, list) else [step]
            msg = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": c["name"],
                        "args": c.get("args", {}),
                        "id": f"mock_{self.pos}_{i}",
                        "type": "tool_call",
                    }
                    for i, c in enumerate(calls)
                ],
            )

        return ChatResult(generations=[ChatGeneration(message=msg)])


def get_llm(responses=None, tool_calls=None):
    """Return a chat model. Groq in 'groq' mode; offline canned mock otherwise."""

    if LLM_MODE == "groq":
        # Live mode needs both a model name and a key; otherwise explain what is missing.
        model = os.getenv("GROQ_MODEL", "").strip()

        if not model or not os.getenv("GROQ_API_KEY", "").strip():
            raise RuntimeError(
                "Groq mode needs GROQ_API_KEY and GROQ_MODEL in .env "
                "(suggested model: openai/gpt-oss-20b; confirm in the Groq console), "
                "or set LLM_MODE=mock."
            )

        # Return the Groq model; temperature 0 and a cache keep replies repeatable.
        print("LIVE model:", model)
        return ChatGroq(
            cache=InMemoryCache(),
            model=model,
            temperature=0,
            http_client=httpx.Client(verify=False),
        )

    # Mock mode: an offline fake model, with scripted tool calls if given, otherwise canned text replies.
    print("MOCK: canned replies")
    if tool_calls:
        return ScriptedToolChat(script=list(tool_calls))

    return FakeListChatModel(responses=list(responses or ["(mock reply)"]))


class TraceLog:
    """Tiny local trace: one dict per step. This class sends nothing anywhere."""

    RATE_PER_1K_TOKENS = 0.0005  # PLACEHOLDER rate in RM per 1,000 tokens (not a real price); use your plan's

    def __init__(self):
        self.rows = []

    def add(self, step, tool="-", tokens_in=0, tokens_out=0, seconds=0.0):
        # Estimate cost from the total tokens at the placeholder rate (not a real price).
        cost = (tokens_in + tokens_out) / 1000 * self.RATE_PER_1K_TOKENS

        # Store one row per step so the trace can be shown as a table.
        self.rows.append(
            {
                "step": step,
                "tool": tool,
                "tokens_in": tokens_in,
                "tokens_out": tokens_out,
                "seconds": round(seconds, 3),
                "est_cost_myr": round(cost, 6),
            }
        )

    def timed(self, step, fn, *args, tool="-", **kw):
        # Run fn, time it, and log a step; token counts are rough guesses (about 4 characters per token).
        t0 = time.perf_counter()
        out = fn(*args, **kw)

        self.add(
            step,
            tool,
            len(str(args)) // 4,
            len(str(out)) // 4,
            time.perf_counter() - t0,
        )

        return out

    # Sum tokens, seconds and estimated cost over all recorded steps.
    def total(self):
        return {
            k: round(sum(r[k] for r in self.rows), 6)
            for k in ("tokens_in", "tokens_out", "seconds", "est_cost_myr")
        }


class TfidfEmbeddings(Embeddings):
    """Offline embeddings: scikit-learn TF-IDF fitted on your corpus, L2-normalised."""

    def __init__(self, corpus):
        # Fit the vocabulary on the corpus once so documents and questions share one vector space.
        self.vec = TfidfVectorizer(norm="l2", lowercase=True).fit(list(corpus))

    def embed_documents(self, texts):
        return self.vec.transform(list(texts)).toarray().tolist()

    def embed_query(self, text):
        return self.embed_documents([text])[0]


# ---- END HELPER ----
# ---- END CANONICAL SETUP ----

Settings file: (outside home) .env (only GROQ_API_KEY, GROQ_MODEL, LLM_MODE, LM_ENV_FILE are read)
Mock mode: LangChain tracing is switched off for this session (nothing is sent anywhere).


## Three rules

1. **Keep your key private.** Never paste it into a notebook, chat or email.
2. **Do not paste personal or company data.** Live mode sends text to a third party (Groq). Use only this course's fictional data.
3. **AI output needs a human check.** A neat answer can still be wrong.

Mock mode (the default) needs no key.

**Practices:** a blank starter looks like `my_attempt = None`. **If you leave it as `None`, the notebook quietly runs the reference answer instead, so Run All works even when you wrote nothing.**</br>
To learn, write your own and set `my_attempt` to it: then yours runs. `check(...)` prints ✔ or ✘ and a hint. `show_solution(step_id)` prints the reference code.
* `USE_SOLUTIONS`: `False` = you type the blanks,
* `USE_SOLUTIONS`: `True` = Run All reads the reference answers.

Reading first, then trying your own, teaches more.

In [4]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotChatMessagePromptTemplate,
)
# Confirm the mode (mock or live) so you know whether replies are canned.
print("LLM_MODE =", LLM_MODE, "| imports ready")

LLM_MODE = mock | imports ready


In [5]:
# Register how to rebuild each Part, so a late learner can catch up with fast_forward.
rebuild("d1-02-A", "d1-02-A")
rebuild("d1-02-B", "d1-02-A", "d1-02-B0", "d1-02-B")

# Reference answer for Step 1: the template text with four holes.
ref(
    None,
    "d1-02-template",
    "Each blank is a name in curly braces.",
    "result = 'Dear {name}, your {item} claim of RM {amount} is {status}.'",
)

# Reference answer for Step 2: the system rule.
ref(
    None,
    "d1-02-system",
    "A system rule says who the model is.",
    "result = 'You are the Seri Maju claims helpdesk. Use only the facts you are given.'",
)

# Remember the Part A claims so a rebuild can recreate them.
state(
    "d1-02-A",
    "CLAIMS = [dict(item='taxi to a client site', amount='35', site='Kuala Lumpur'),\n"
    "         dict(item='laptop repair', amount='1,400', site='Penang'),\n"
    "         dict(item='customer dinner', amount='3,800', site='Johor Bahru')]",
)

# Reference answer for the practice: a claim prompt that names all three holes.
ref(
    "d1-02-A",
    "d1-02-prompt",
    "Name all three holes in the human message.",
    "result = ChatPromptTemplate.from_messages([\n"
    "    ('system', 'You are the Seri Maju claims helpdesk. Use only the facts given.'),\n"
    "    ('human', 'Claim: {item}, RM {amount} ({site}). Who should approve it?')])\n"
    "claim_prompt = result",
)

# Remember the shape check used in Part B.
state(
    "d1-02-B",
    "def fits(text):\n"
    "    return text.strip().startswith('Approver:') and '| Policy:' in text",
)

# Reference answer for the few-shot prompt: examples sit between the rule and the question.
ref(
    "d1-02-B",
    "d1-02-fewshot",
    "Examples are human + ai pairs first.",
    "example_prompt = ChatPromptTemplate.from_messages(\n"
    "    [('human', '{ticket}'), ('ai', '{label}')]\n)\n"
    "shots = FewShotChatMessagePromptTemplate(\n"
    "    example_prompt=example_prompt, examples=EXAMPLES\n)\n"
    "result = ChatPromptTemplate.from_messages(\n"
    "    [('system', SYSTEM), shots, ('human', '{ticket}')]\n)\n"
    "few_prompt = result",
)

# Reference verdict, backed by the fit table rather than a feeling.
ref(
    None,
    "d1-02-verdict",
    "The table, not a feeling, backs the verdict.",
    "result = dict(verdict='better',\n"
    "              reason='All three few-shot replies fit; none of zero-shot did.')",
)

# Remember the Part B data and zero-shot prompt so a rebuild can recreate them.
state(
    "d1-02-B0",
    "SYSTEM = 'You sort expense claims for Seri Maju Berhad. Reply in the shape shown.'\n"
    "EXAMPLES = [\n"
    "    {'ticket': 'Claim: parking at a supplier visit, RM 80 (Johor Bahru).',\n"
    "     'label': 'Approver: Line manager | Policy: SM-POL-03'},\n"
    "    {'ticket': 'Claim: team workshop snacks, RM 2,000 (Kuala Lumpur).',\n"
    "     'label': 'Approver: Department head | Policy: SM-POL-03'},\n"
    "    {'ticket': 'Claim: conference booth, RM 4,500 (Penang).',\n"
    "     'label': 'Approver: Finance Director | Policy: SM-POL-03'}\n]\n"
    "TICKETS = ['Claim: ' + c['item'] + ', RM ' + c['amount'] + ' (' + c['site'] + ').'\n"
    "           for c in CLAIMS]\n"
    "zero_prompt = ChatPromptTemplate.from_messages(\n"
    "    [('system', SYSTEM), ('human', '{ticket}')]\n)",
)

In [6]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-groq>=0.3,<0.4" "langchain-text-splitters>=0.3,<0.4" "langgraph>=0.6,<0.7" "pydantic>=2,<3" "scikit-learn>=1.4" "python-dotenv"

In [7]:
import sys
import langchain
import langchain_core
import langchain_groq

print("Python:", sys.version)
print("LangChain:", getattr(langchain, "__version__", "unknown"))
print("LangChain Core:", getattr(langchain_core, "__version__", "unknown"))
print("LangChain Groq:", getattr(langchain_groq, "__version__", "unknown"))
print("LangChain file:", langchain.__file__)
print("Has verbose:", hasattr(langchain, "verbose"))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
LangChain: 0.3.30
LangChain Core: 0.3.86
LangChain Groq: 0.3.8
LangChain file: /usr/local/lib/python3.12/dist-packages/langchain/__init__.py
Has verbose: True


In [8]:
print("DATA:", "DATA" in globals())
print("get_llm:", "get_llm" in globals())
print("pick:", "pick" in globals())
print("check:", "check" in globals())
print("USE_SOLUTIONS:", "USE_SOLUTIONS" in globals())

DATA: False
get_llm: True
pick: True
check: True
USE_SOLUTIONS: True


## Templates and Roles

### Idea: A Template is a Prompt with Blanks (●)

A **prompt** is the text you send to a model.

A **template** is a prompt with holes/placeholders written in curly braces, like `{name}`. You fill the holes each time, like a mail-merge letter (see the picture at the top). One good template gives the same shape every time, so the answers are easier to check.

### Idea: Chat Roles (●)

A chat model reads a **list of messages**. Each message has a **role**:

| Role | Who speaks | Use it for |
|---|---|---|
| `system` | You, the builder | Standing rules: who the model is, what it may use |
| `human` | The staff member | The question or claim of the moment |
| `ai` or `assistant` | The model | Its earlier replies (and, later, worked examples) |

**Step 1 - fill a template (●)**

Write a template string with four holes: `name`, `item`, `amount` and `status`. Then it is filled for you.

In [9]:
my_text = None  # TODO: a string with {name}, {item}, {amount} and {status}

# Use your template, or the reference one when you left the blank as None.
template_text = pick("d1-02-template", my_text, is_using_solutions=USE_SOLUTIONS)
# Turn the text into a reusable template object.
tmpl = PromptTemplate.from_template(template_text)

# Fill every hole once to see the finished letter.
letter = tmpl.format(name="Aina", item="taxi", amount="35", status="approved")
print(letter)

# Self-check: the template must have exactly the four expected holes.
check(
    "d1-02-template",
    set(tmpl.input_variables) == {"name", "item", "amount", "status"},
    "Use exactly four holes: {name} {item} {amount} {status}.",
)

Dear Aina, your taxi claim of RM 35 is approved.
✔ d1-02-template: looks right


True

**Step 2 - a system rule (●)**

Write one sentence that tells the model who it is and what it may use. The chat template is built for you with the rule as the `system` message and the question as the `human` message.

In [10]:
my_rule = None  # TODO: one sentence of at least 6 words

# Use your rule, or the reference rule when you left the blank as None.
system_rule = pick("d1-02-system", my_rule, is_using_solutions=USE_SOLUTIONS)

# Put the standing rule in a system message and the question in a human message.
chat = ChatPromptTemplate.from_messages(
    [("system", system_rule), ("human", "{question}")]
)

# Fill the question hole and keep the resulting list of role-tagged messages.
msgs = chat.invoke({"question": "Who approves RM 700?"}).to_messages()

# Show each message with its role, to see who says what.
for m in msgs:
    print(f"[{m.type}] {m.content}")

# Self-check: one system message, one human message, and a rule of enough words.
check(
    "d1-02-system",
    [m.type for m in msgs] == ["system", "human"] and len(system_rule.split()) >= 6,
    "Write a system rule of 6 or more words.",
)

[system] You are the Seri Maju claims helpdesk. Use only the facts you are given.
[human] Who approves RM 700?
✔ d1-02-system: looks right


True

**Practice - your own claim prompt (●)**

Finish the human message so it uses all three holes: `{item}`, `{amount}` and `{site}`. Then the cell asks the (mock) model about three claims. No coding? Leave `my_attempt = None`.

In [11]:
# Three fictional claims reused for the rest of the notebook.
CLAIMS = [
    dict(item="taxi to a client site", amount="35", site="Kuala Lumpur"),
    dict(item="laptop repair", amount="1,400", site="Penang"),
    dict(item="customer dinner", amount="3,800", site="Johor Bahru"),
]

# Practice: build a claim prompt whose human message uses all three holes.
my_prompt = ChatPromptTemplate.from_messages(
    [("system", "You are the Seri Maju claims helpdesk. Use only the facts given.")]
    + SYSTEM_PROMPT
    + [("human", "____")]  # TODO: mention {item}, {amount} and {site}
)

my_attempt = None  # when ready, write: my_attempt = my_prompt

# Use your prompt, or the reference prompt when you left the attempt as None.
claim_prompt = pick("d1-02-prompt", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Self-check: the prompt must ask for exactly item, amount and site.
ok = set(claim_prompt.input_variables) == {"item", "amount", "site"}
check("d1-02-prompt", ok, "The human message needs {item}, {amount} and {site}.")

✔ d1-02-prompt: looks right


True

In [12]:
# Canned replies, one per claim, so the demo runs without a key and is repeatable.
canned = [
    "Your line manager can approve this claim.",
    "Ask your department head to approve it.",
    "It needs Finance Director approval.",
]

# Fake model that returns the canned replies in order.
llm = get_llm(responses=canned)

# Run each claim through prompt then model, and print the routed reply beside its amount.
for claim in CLAIMS:
    reply = llm.invoke(claim_prompt.invoke(claim))  # prompt first, then model
    print(f"RM {claim['amount']:>6} {claim['site']:<13} -> {reply.content}")

MOCK: canned replies
RM     35 Kuala Lumpur  -> Your line manager can approve this claim.
RM  1,400 Penang        -> Ask your department head to approve it.
RM  3,800 Johor Bahru   -> It needs Finance Director approval.


### Check yourself

1. What is the difference between a prompt and a template? <details><summary>Answer:</br></br></summary>A prompt is the text sent to the model. A template is a prompt with holes you fill each time.</br></br></details>
2. Which role holds the standing rules? <details><summary>Answer:</br></br></summary>`system`.</br></br></details>
3. What happens if you call `claim_prompt.invoke({'item': 'taxi'})` and leave out a hole? <details><summary>Answer:</br></br></summary>You get an error about the missing variable. That is helpful: the mistake shows up before the model is called.</br></br></details>
4. The mock replies above were canned. What does that prove? <details><summary>Answer:</br></br></summary>Only that the plumbing works. It says nothing about how a real model behaves.</br></br></details>

To continue you need `CLAIMS` and `claim_prompt`.

In [13]:
# Part B needs the claims and prompt from Part A, so check they exist.
need = ["CLAIMS", "claim_prompt"]
missing = [n for n in need if n not in globals()]

# Report whether anything is missing.
print("All set for Part B." if not missing else "Missing: " + ", ".join(missing))

# If Part A never ran, point to the way back instead of guessing.
if missing and "d1-02-A" not in CODES:
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

# If only the names are missing, rebuild them from reference code.
elif missing:
    fast_forward("d1-02-A")  # rebuilds the names above from reference code

# Confirm the prompt works: it should yield a system message plus a human message.
n = len(claim_prompt.invoke(CLAIMS[0]).to_messages())
print("claim_prompt makes", n, "messages for one claim")

All set for Part B.
claim_prompt makes 2 messages for one claim


## Examples Teach the Shape

### Idea: Zero-Shot and Few-Shot (●)

- **Zero-shot:** you give only an instruction. The model guesses the format, and often chats.
- **Few-shot:** you also show a few worked examples (the *shots*) as earlier human and `assistant` messages. The model copies their shape.
- Our target shape: `Approver: <who> | Policy: SM-POL-<number>`. The claims policy (fictional) says up to RM 500: line manager; RM 501 to RM 3,000: department head; above RM 3,000: Finance Director.

> ! **MOCK WARNING**
> The replies below are canned. The zero-shot ones are chatty on purpose and the few-shot ones are tidy on purpose.
> This shows how to *measure* the difference. It does not prove few-shot always wins. With a live model you must run it and look.

**Step 1 - the data and the zero-shot prompt (●).**

Run this cell as it is. The three shots are not the three test claims.

In [14]:
# The standing rule shared by the zero-shot and few-shot prompts.
SYSTEM = "You sort expense claims for Seri Maju Berhad. Reply in the shape shown."

# Worked examples (the shots). They differ from the test claims on purpose.
EXAMPLES = [
    {
        "ticket": "Claim: parking at a supplier visit, RM 80 (Johor Bahru).",
        "label": "Approver: Line manager | Policy: SM-POL-03",
    },
    {
        "ticket": "Claim: team workshop snacks, RM 2,000 (Kuala Lumpur).",
        "label": "Approver: Department head | Policy: SM-POL-03",
    },
    {
        "ticket": "Claim: conference booth, RM 4,500 (Penang).",
        "label": "Approver: Finance Director | Policy: SM-POL-03",
    },
]

# Turn each test claim into the same one-line ticket text.
TICKETS = [
    "Claim: " + c["item"] + ", RM " + c["amount"] + " (" + c["site"] + ")."
    for c in CLAIMS
]

# Zero-shot prompt: the instruction only, with no examples.
zero_prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM)] + SYSTEM_PROMPT + [("human", "{ticket}")]
)


def fits(text):  # the quick shape check: does the reply look like our target?
    return text.strip().startswith("Approver:") and "| Policy:" in text


# Look at the tickets that will be sent to the model.
print(TICKETS)

['Claim: taxi to a client site, RM 35 (Kuala Lumpur).', 'Claim: laptop repair, RM 1,400 (Penang).', 'Claim: customer dinner, RM 3,800 (Johor Bahru).']


**Step 2 - build the few-shot prompt (●).**

Wrap the examples in `FewShotChatMessagePromptTemplate`, then place it between the system message and the final question.

In [15]:
# How one worked example is laid out: the ticket as human, the label as the model's answer.
example_prompt = ChatPromptTemplate.from_messages(
    [("human", "{ticket}"), ("ai", "{label}")]
)

# Practice: wrap the examples so they can sit inside a chat prompt.
my_shots = None  # TODO: FewShotChatMessagePromptTemplate(example_prompt, examples)

# Build the few-shot prompt: rule first, then your examples, then the real question.
my_few = ChatPromptTemplate.from_messages(
    [("system", SYSTEM)]
    + SYSTEM_PROMPT
    + [my_shots or ("human", "TODO"), ("human", "{ticket}")]
)

my_attempt = None  # when ready, write: my_attempt = my_few

# Use your prompt, or the reference prompt when you left the attempt as None.
few_prompt = pick("d1-02-fewshot", my_attempt, is_using_solutions=USE_SOLUTIONS)

# Fill in one ticket to see the messages the model would receive.
msgs = few_prompt.invoke({"ticket": TICKETS[0]}).to_messages()

# Show the order of roles.
print([m.type for m in msgs])

# Self-check: one system message, a human and ai pair per shot, then the question.
ok = len(msgs) == 1 + 2 * len(EXAMPLES) + 1 and msgs[-1].content == TICKETS[0]
check("d1-02-fewshot", ok, "Expect system + (human, ai) per shot + the question.")

['system', 'human', 'ai', 'human', 'ai', 'human', 'ai', 'human']
✔ d1-02-fewshot: looks right


True

**Step 3 - run both on the three claims (CORE ●).** Run the cell as it is and read the table. The shape check is `fits`, a plain function. The words *fits* and *does not fit* are used, not colour.

In [16]:
# The tidy replies we want from the few-shot run.
EXPECTED = [
    "Approver: Line manager | Policy: SM-POL-03",
    "Approver: Department head | Policy: SM-POL-03",
    "Approver: Finance Director | Policy: SM-POL-03",
]

# Chatty replies standing in for a zero-shot model.
CHATTY = [
    "A taxi fare is small, so your line manager can approve it.",
    "Probably your department head; check the claims policy.",
    "That is a big dinner. Finance will want to look at it.",
]  # canned

# Two fake models so the comparison is repeatable without an API key.
zero_llm, few_llm = get_llm(responses=CHATTY), get_llm(responses=EXPECTED)


# Helper that sends one ticket through a prompt and model and returns the reply text.
def ask(prompt, llm, ticket):
    return llm.invoke(prompt.invoke({"ticket": ticket})).content


# Run all three tickets both ways, ready to be compared in the next cell.
zero_out = [ask(zero_prompt, zero_llm, t) for t in TICKETS]
few_out = [ask(few_prompt, few_llm, t) for t in TICKETS]

MOCK: canned replies
MOCK: canned replies


In [17]:
# Show the verdict in words, not colour.
def word(ok):
    return "fits" if ok else "does not fit"


# Compare the two approaches ticket by ticket using the shape check.
for i in range(3):
    z, f = word(fits(zero_out[i])), word(fits(few_out[i]))
    print(f"{i + 1}. zero-shot {z:<13}| few-shot {f}")

# Print the few-shot replies so you can read what actually fits.
print("few-shot replies:", few_out, sep="\n")

1. zero-shot does not fit | few-shot fits
2. zero-shot does not fit | few-shot fits
3. zero-shot does not fit | few-shot fits
few-shot replies:
['Approver: Line manager | Policy: SM-POL-03', 'Approver: Department head | Policy: SM-POL-03', 'Approver: Finance Director | Policy: SM-POL-03']


**Practice - your verdict (●, no coding needed).**

Is few-shot *better*, the *same* or *worse* than zero-shot here?

Give one reason in your own words. Leave `my_attempt = None` to see an example.

In [18]:
my_verdict = None  # replace None with: dict(verdict='better', reason='your reason')

# Use your verdict, or the reference verdict when you left it as None.
v = pick("d1-02-verdict", my_verdict, is_using_solutions=USE_SOLUTIONS)

# Count how many replies fit in each approach and let the numbers give the verdict.
zero_n, few_n = sum(map(fits, zero_out)), sum(map(fits, few_out))
table_says = "better" if few_n > zero_n else "same" if few_n == zero_n else "worse"

# Put your verdict beside what the table says.
print("Your verdict:", v["verdict"], "| the table says:", table_says)
print("Fit counts: zero-shot", zero_n, "of 3, few-shot", few_n, "of 3")

# Self-check: a valid verdict plus a short reason.
check(
    "d1-02-verdict",
    v["verdict"] in ("better", "same", "worse") and len(v["reason"].split()) >= 5,
    "Choose better/same/worse and write 5+ words.",
)

Your verdict: better | the table says: better
Fit counts: zero-shot 0 of 3, few-shot 3 of 3
✔ d1-02-verdict: looks right


True

### Check yourself

1. What is the difference between zero-shot and few-shot? <details><summary>Answer:</br></br></summary>Zero-shot gives only an instruction. Few-shot adds worked examples the model can copy.</br></br></details>
2. Why must the shots not be the same as the test claims? <details><summary>Answer:</br></br></summary>Otherwise the model could just repeat an answer it was shown, and you would learn nothing about new claims.</br></br></details>
3. Our replies were canned. What would you do with a live model? <details><summary>Answer:</br></br></summary>Run it on the same claims, apply the same `fits` check and compare. Do not assume few-shot wins.</br></br></details>
4. A reply fits the shape but names the wrong approver. Is it right? <details><summary>Answer:</br></br></summary>No. A shape check does not check facts. A human or a fact check still has to look.</br></br></details>

You should have `zero_prompt`, `few_prompt`, `fits` and `TICKETS`.

In [19]:
# Confirm everything from this notebook is in place for the next one.
need = ["zero_prompt", "few_prompt", "fits", "TICKETS"]
missing = [n for n in need if n not in globals()]

# Report whether anything is missing.
print("d1-02 complete." if not missing else "Missing: " + ", ".join(missing))

# If a Part never ran, point to the way back instead of guessing.
if missing and not all(p in CODES for p in ("d1-02-A", "d1-02-B")):
    print("Click the heading Part A, Run > Run All Above Selected Cell, then rerun.")

# If only the names are missing, rebuild them from reference code.
elif missing:
    fast_forward("d1-02-B")  # rebuilds Part A and Part B names from reference code

# Confirm the few-shot prompt works end to end.
n = len(few_prompt.invoke({"ticket": TICKETS[0]}).to_messages())
print("few_prompt messages for one claim:", n)

d1-02 complete.
few_prompt messages for one claim: 8


## Recap and next step

- A **template** is a prompt with holes. A **chat template** adds roles: `system` for rules, `human` for the question, `assistant` for examples.
- **Few-shot** shows worked examples. Measure the difference with a small check, never by feel alone.
- Mock replies are canned: they test your plumbing, not the model.
- Next: **[d1-03-trusted-answers-validate-repair-critique.ipynb](d1-03-trusted-answers-validate-repair-critique.ipynb)** (check structured answers, repair them once, and improve a draft with a budget).

### The pipe form (◆)

LangChain lets you join steps with `|`. `a | b` means: feed the output of `a` into `b`. So `prompt | llm | StrOutputParser()` is the same as the three steps you did by hand, and it returns plain text.

### A stricter format check (◆)

A **regular expression** (regex) describes a text pattern. The second half of the next cell checks the whole line, not just the start.

In [21]:
# Regex is used for the stricter whole-line format check below.
import re
from langchain_core.output_parsers import StrOutputParser

# Join prompt, model and parser into one pipe that returns plain text.
chain = claim_prompt | get_llm(responses=["Line manager."]) | StrOutputParser()

# Run the pipe and confirm the output is a plain string.
print(type(chain.invoke(CLAIMS[0])).__name__, "->", chain.invoke(CLAIMS[0]))

# Pattern for the exact target line, checked from start to end.
FORMAT = re.compile(r"^Approver: [A-Za-z ]+ [|] Policy: SM-POL-[0-9]{2}$")


# Wrap the pattern as a yes/no check.
def fits_format(text):
    return bool(FORMAT.match(text.strip()))


# Test the check on one tidy line and one chatty line.
ok = fits_format("Approver: Department head | Policy: SM-POL-03") and not fits_format(
    "Probably your department head."
)

# Self-check: the tidy line passes and the chatty line fails.
check("d1-02-format", ok, "Accept the tidy line and reject the chatty one.")

MOCK: canned replies
str -> Line manager.
✔ d1-02-format: looks right


True

### Notes for later (■)

- **Extra templates.** Write a template for a leave-reply and one for a travel-approval reply. Add a fourth hole, `{tone}`, to the system message.
- **More example sets.** Add a fourth shot for a claim that fits no policy, such as a request about the staff cafeteria, with the label `Approver: none | Policy: none`. Does the shape check still work?
- **Honesty labels.** Mock replies are canned. Policy figures are fictional (see `data/workplace_corpus/sm-pol-03-expense-claims.txt`). Not legal or tax advice.